<a href="https://colab.research.google.com/github/23AD083/Emotion_Drift_Detection_using_NLP/blob/main/NLP_PROJECT_EMOTION_DRIFT_DETECTION__streamlit(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets scikit-learn pandas streamlit pyngrok

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd

# Define the base path where your MELD dataset files are located
# For example, if they are in 'My Drive/data/'
# Replace 'path/to/your/meld_files/' with the actual path in your Google Drive
path_to_meld_files = '/content/drive/MyDrive/nlp_emotion_drift/'

# LOAD MELD
train = pd.read_csv(f"{path_to_meld_files}train_sent_emo.csv")
dev = pd.read_csv(f"{path_to_meld_files}dev_sent_emo.csv")
test = pd.read_csv(f"{path_to_meld_files}test_sent_emo.csv")

meld = pd.concat([train, dev, test])

meld = meld.rename(columns={
    "Utterance":"utterance",
    "Emotion":"emotion"
})

meld = meld[["utterance","emotion"]]


# Define the path for the DailyDialog toy dataset
# Replace 'path/to/your/dailydialog_files/' with the actual path in your Google Drive
path_to_dailydialog_files = '/content/drive/MyDrive/nlp_emotion_drift/'

# LOAD DAILYDIALOG
data = []

with open(f"{path_to_dailydialog_files}toy_train.txt","r",encoding="utf-8") as f:
    lines = f.readlines()

for line in lines:

    if line.strip()=="":
        continue

    data.append({
        "utterance":line.strip(),
        "emotion":"neutral"
    })

dailydialog = pd.DataFrame(data)


# COMBINE
df = pd.concat([meld,dailydialog])

print("Dataset size:",df.shape)

Dataset size: (23708, 2)


In [6]:
import re

def clean_text(text):

    text = text.lower()
    text = re.sub(r"http\S+","",text)
    text = re.sub(r"[^a-zA-Z\s]","",text)

    return text


df["utterance"] = df["utterance"].apply(clean_text)

df = df.dropna()

In [7]:
print(df["emotion"].value_counts())

min_count = df["emotion"].value_counts().min()

df = df.groupby("emotion").sample(min_count)

print(df["emotion"].value_counts())

emotion
neutral     16436
joy          2308
surprise     1636
anger        1607
sadness      1002
disgust       361
fear          358
Name: count, dtype: int64
emotion
anger       358
disgust     358
fear        358
joy         358
neutral     358
sadness     358
surprise    358
Name: count, dtype: int64


In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["emotion"])

In [9]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["utterance"],
    df["label"],
    test_size=0.1,
    random_state=42
)

In [10]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [11]:
import torch

class EmotionDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


train_dataset = EmotionDataset(train_encodings, list(train_labels))
val_dataset = EmotionDataset(val_encodings, list(val_labels))

In [12]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(le.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./emotion_model",

    num_train_epochs=2,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    learning_rate=3e-5,

    # evaluation_strategy="epoch", # Commented out again to fix the TypeError

    # save_strategy="epoch", # Commented out again to fix the TypeError

    logging_steps=100
)

In [14]:
from sklearn.metrics import accuracy_score,f1_score

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels,preds)
    f1 = f1_score(labels,preds,average="weighted")

    return {
        "accuracy":acc,
        "f1":f1
    }

In [15]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

Step,Training Loss
100,1.810295
200,1.587610


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=282, training_loss=1.6421158432115055, metrics={'train_runtime': 39.693, 'train_samples_per_second': 113.622, 'train_steps_per_second': 7.105, 'total_flos': 85187754827340.0, 'train_loss': 1.6421158432115055, 'epoch': 2.0})

In [17]:
print(len(train_dataset))


2255


In [18]:
print(trainer.train_dataset)

In [19]:
trainer.save_model("emotion_model")

tokenizer.save_pretrained("emotion_model")

import pickle

with open("label_encoder.pkl","wb") as f:
    pickle.dump(le,f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
emotion_score = {

"joy":3,
"surprise":2,
"neutral":1,
"sadness":-1,
"fear":-2,
"disgust":-3

}

def detect_drift(emotions):

    drift=[]

    for i in range(1,len(emotions)):

        prev=emotion_score[emotions[i-1]]
        curr=emotion_score[emotions[i]]

        if abs(curr-prev)>=2:

            drift.append(i+1)

    return drift

In [21]:
!pip install plotly

In [22]:
emotion_score = {
    "joy": 3,
    "surprise": 2,
    "neutral": 1,
    "sadness": -1,
    "fear": -2,
    "disgust": -3
}

In [23]:
def detect_drift(emotions):

    drift = []

    for i in range(1,len(emotions)):
        prev = emotion_score.get(emotions[i-1],0)
        curr = emotion_score.get(emotions[i],0)
        if abs(curr-prev) >= 2:
            drift.append(i+1)

    return drift

In [24]:
pip install streamlit transformers torch scikit-learn plotly

In [25]:
# Replace 'YOUR_AUTHTOKEN' with your actual ngrok authtoken
!ngrok authtoken 3AkGxjQjHQqRSwQZLZOZ8DzR0qN_6pYcZTKzRbT3zXaH5qs87

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [37]:
import streamlit as st
import torch
import pickle
import plotly.graph_objects as go
import os
import subprocess
import time

from pyngrok import ngrok
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


# ----------------------------
# NGROK AUTH (UNCHANGED)
# ----------------------------

ngrok.set_auth_token("3AkGxjQjHQqRSwQZLZOZ8DzR0qN_6pYcZTKzRbT3zXaH5qs87")


# ----------------------------
# STREAMLIT APP CODE
# ----------------------------

streamlit_app_code = """

import streamlit as st
import torch
import pickle
import plotly.graph_objects as go
import torch.nn.functional as F

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


# ----------------------------
# LOAD MODEL
# ----------------------------

tokenizer = DistilBertTokenizer.from_pretrained("emotion_model")
model = DistilBertForSequenceClassification.from_pretrained("emotion_model")

with open("label_encoder.pkl","rb") as f:
    le = pickle.load(f)


emotion_score = {
    "joy":3,
    "surprise":2,
    "neutral":1,
    "sadness":-1,
    "fear":-2,
    "anger":-3
}


# ----------------------------
# SAMPLE TEST INPUTS
# ----------------------------

st.title("🤖 Emotion Drift Detection Chatbot")

st.subheader("🧪 Sample Test Inputs")

samples = {
    "Happy → Angry Drift":
    '''I was very happy today
My code worked perfectly
Suddenly my laptop crashed
Now I feel very angry''',

    "Angry → Happy Recovery":
    '''My project was not working
I felt very frustrated
My friend helped me debug
Now everything works
I feel very happy''',

    "Stable Positive Mood":
    '''Today was a great day
I enjoyed working on my project
Everything went smoothly
I feel satisfied''',

    "Mental Stress Pattern":
    '''I feel very sad today
Nothing seems to work
I feel anxious about my future
Everything feels overwhelming''',

    "Mixed Emotion Conversation":
    '''I was excited to start my project
But I faced many bugs
I got very angry and frustrated
Later I solved the issue
Now I feel relieved'''
}

sample_choice = st.selectbox("Choose a sample test", list(samples.keys()))

st.text_area(
    "Sample Conversation (copy and paste into chat)",
    samples[sample_choice],
    height=150
)

st.divider()


# ----------------------------
# USER LOGIN
# ----------------------------

st.sidebar.title("User Login")

user_id = st.sidebar.text_input("Enter User ID","user1")


# ----------------------------
# SESSION STORAGE
# ----------------------------

if "users" not in st.session_state:
    st.session_state.users = {}

if user_id not in st.session_state.users:
    st.session_state.users[user_id] = []

if "current_conv" not in st.session_state:
    st.session_state.current_conv = None


# ----------------------------
# CREATE NEW CONVERSATION
# ----------------------------

if st.sidebar.button("New Conversation"):

    conv = {
        "messages":[],
        "emotions":[],
        "confidence":[]
    }

    st.session_state.users[user_id].append(conv)
    st.session_state.current_conv = len(st.session_state.users[user_id]) - 1


# ----------------------------
# SELECT CONVERSATION
# ----------------------------

conversations = st.session_state.users[user_id]

if len(conversations) > 0:

    conv_index = st.sidebar.selectbox(
        "Select Conversation",
        range(len(conversations)),
        format_func=lambda x: f"Conversation {x+1}"
    )

    st.session_state.current_conv = conv_index


# ----------------------------
# EMOTION PREDICTION
# ----------------------------

def predict_emotion(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs).item()

    confidence = torch.max(probs).item()

    emotion = le.inverse_transform([pred])[0]

    return emotion, confidence


# ----------------------------
# DRIFT DETECTION
# ----------------------------

def detect_drift(emotions):

    drift = []

    for i in range(1,len(emotions)):

        prev = emotion_score.get(emotions[i-1],0)
        curr = emotion_score.get(emotions[i],0)

        if abs(curr-prev) >= 2:
            drift.append(i+1)

    return drift


# ----------------------------
# CHAT UI
# ----------------------------

if st.session_state.current_conv is None:

    st.info("Create a new conversation from sidebar 👈")

else:

    conv = conversations[st.session_state.current_conv]

    user_input = st.chat_input("Type your message")

    if user_input:

        emotion, confidence = predict_emotion(user_input)

        reply = f"Emotion: **{emotion}** | Confidence: **{round(confidence*100,2)}%**"

        conv["messages"].append({"role":"user","content":user_input})
        conv["messages"].append({"role":"assistant","content":reply})

        conv["emotions"].append(emotion)
        conv["confidence"].append(confidence)


    # DISPLAY CHAT

    for msg in conv["messages"]:
        st.chat_message(msg["role"]).write(msg["content"])


    # ----------------------------
    # CONVERSATION ANALYTICS
    # ----------------------------

    if len(conv["emotions"]) > 0:

        st.divider()
        st.header("Conversation Analytics")

        emotions = conv["emotions"]
        confidence = conv["confidence"]

        drift = detect_drift(emotions)

        st.write("Predicted Emotions:", emotions)
        st.write("Confidence:", [round(c*100,2) for c in confidence])

        if len(drift) > 0:
            st.warning(f"⚠ Emotion Drift Detected at turns: {drift}")


        scores = [emotion_score[e] for e in emotions]

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=list(range(1,len(scores)+1)),
            y=scores,
            mode="lines+markers",
            name="Emotion Trend"
        ))

        st.plotly_chart(fig, key=f"conv_chart_{st.session_state.current_conv}")


        # Emotion Change Analysis

        if len(emotions) > 1:

            st.subheader("Emotion Changes")

            for i in range(1,len(emotions)):

                prev = emotions[i-1]
                curr = emotions[i]

                if prev != curr:
                    st.write(f"Turn {i} → {i+1} : {prev} ➜ {curr}")


    # ----------------------------
    # MENTAL HEALTH ANALYSIS
    # ----------------------------

    if len(conv["emotions"]) > 3:

        negative_count = sum(
            1 for e in conv["emotions"]
            if e in ["sadness","fear","anger"]
        )

        if negative_count >= len(conv["emotions"])/2:

            st.error(\"\"\"

⚠ Mental Health Alert

Your conversation shows a strong negative emotional trend.

Suggestions:

• Take a short break
• Talk to a trusted friend
• Try relaxation techniques

\"\"\")

        else:

            st.success("✅ Emotional state appears stable.")


# ----------------------------
# GLOBAL DASHBOARD
# ----------------------------

st.divider()
st.header("📊 All User Conversations Dashboard")

for user, convs in st.session_state.users.items():

    st.subheader(f"User: {user}")

    for i, conv in enumerate(convs):

        with st.expander(f"Conversation {i+1}"):

            if len(conv["emotions"]) == 0:
                st.write("No messages yet")

            else:

                scores = [emotion_score[e] for e in conv["emotions"]]

                fig = go.Figure()

                fig.add_trace(go.Scatter(
                    x=list(range(1,len(scores)+1)),
                    y=scores,
                    mode="lines+markers"
                ))

                st.plotly_chart(fig, key=f"dashboard_{user}_{i}")

                drift = detect_drift(conv["emotions"])

                if len(drift) > 0:
                    st.warning(f"Drift detected at turns: {drift}")
"""


# Write the Streamlit app code
with open("app.py", "w") as f:
    f.write(streamlit_app_code)


# ----------------------------
# START STREAMLIT
# ----------------------------

if "STREAMLIT_SERVER_RUNNING" not in os.environ:
    os.environ["STREAMLIT_SERVER_RUNNING"] = "1"

    subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port=8501"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(5)


# ----------------------------
# CREATE PUBLIC LINK
# ----------------------------

public_url = ngrok.connect(8501)

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://delly-transnational-alda.ngrok-free.dev" -> "http://localhost:8501"


In [38]:
!streamlit run app.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.253.227:8501

Loading weights: 100% 104/104 [00:00<00:00, 1033.65it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104 [00:00<00:00, 2004.46it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104 [00:00<00:00, 1060.81it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104 [00:00<00:00, 803.22it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104 [00:00<00:00, 883.60it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104 [00:00<00:00, 1910.97it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104 [00:00<00:00, 996.20it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104 [00:00<00:00, 944.18it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 104/104

In [39]:
import gradio as gr
import torch
import pickle
import plotly.graph_objects as go
import torch.nn.functional as F

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


# ----------------------------
# LOAD MODEL
# ----------------------------

tokenizer = DistilBertTokenizer.from_pretrained("emotion_model")
model = DistilBertForSequenceClassification.from_pretrained("emotion_model")

with open("label_encoder.pkl","rb") as f:
    le = pickle.load(f)


emotion_score = {
    "joy":3,
    "surprise":2,
    "neutral":1,
    "sadness":-1,
    "fear":-2,
    "anger":-3
}


# ----------------------------
# EMOTION PREDICTION
# ----------------------------

def predict_emotion(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs).item()

    confidence = torch.max(probs).item()

    emotion = le.inverse_transform([pred])[0]

    return emotion, confidence


# ----------------------------
# DRIFT DETECTION
# ----------------------------

def detect_drift(emotions):

    drift = []

    for i in range(1,len(emotions)):

        prev = emotion_score.get(emotions[i-1],0)
        curr = emotion_score.get(emotions[i],0)

        if abs(curr-prev) >= 2:
            drift.append(i+1)

    return drift


# ----------------------------
# MAIN ANALYSIS FUNCTION
# ----------------------------

def analyze_conversation(conversation):

    lines = conversation.split("\n")

    emotions = []
    confidence = []

    for line in lines:

        if line.strip() != "":
            emo, conf = predict_emotion(line)

            emotions.append(emo)
            confidence.append(round(conf*100,2))


    drift = detect_drift(emotions)

    scores = [emotion_score[e] for e in emotions]


    # Plot graph
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=list(range(1,len(scores)+1)),
        y=scores,
        mode="lines+markers",
        name="Emotion Trend"
    ))


    # Mental Health Analysis
    negative_count = sum(
        1 for e in emotions if e in ["sadness","fear","anger"]
    )

    if len(emotions) > 3 and negative_count >= len(emotions)/2:

        mental_status = """
⚠ Mental Health Alert

Your conversation shows a strong negative emotional trend.

Suggestions:
• Take a short break
• Talk to a trusted friend
• Try relaxation techniques
"""

    else:
        mental_status = "✅ Emotional state appears stable."


    return emotions, confidence, drift, fig, mental_status


# ----------------------------
# SAMPLE INPUTS
# ----------------------------

samples = {
"Happy → Angry Drift":
"""I was very happy today
My code worked perfectly
Suddenly my laptop crashed
Now I feel very angry""",

"Angry → Happy Recovery":
"""My project was not working
I felt very frustrated
My friend helped me debug
Now everything works
I feel very happy""",

"Stable Positive Mood":
"""Today was a great day
I enjoyed working on my project
Everything went smoothly
I feel satisfied""",

"Mental Stress Pattern":
"""I feel very sad today
Nothing seems to work
I feel anxious about my future
Everything feels overwhelming""",

"Mixed Emotion Conversation":
"""I was excited to start my project
But I faced many bugs
I got very angry and frustrated
Later I solved the issue
Now I feel relieved"""
}


# ----------------------------
# GRADIO UI
# ----------------------------

with gr.Blocks(title="Emotion Drift Detection") as demo:

    gr.Markdown("# 🤖 Emotion Drift Detection Chatbot")

    gr.Markdown("Analyze emotional patterns and detect emotional drift in conversations.")

    sample_choice = gr.Dropdown(
        list(samples.keys()),
        label="Choose Sample Test"
    )

    sample_text = gr.Textbox(
        label="Sample Conversation",
        lines=6
    )

    sample_choice.change(
        lambda x: samples[x],
        sample_choice,
        sample_text
    )


    conversation_input = gr.Textbox(
        label="Enter Conversation (one sentence per line)",
        lines=8
    )

    analyze_btn = gr.Button("Analyze Emotion")


    emotion_output = gr.JSON(label="Predicted Emotions")

    confidence_output = gr.JSON(label="Confidence (%)")

    drift_output = gr.JSON(label="Emotion Drift Points")

    graph_output = gr.Plot(label="Emotion Trend Graph")

    mental_output = gr.Textbox(label="Mental Health Analysis")


    analyze_btn.click(
        analyze_conversation,
        inputs=conversation_input,
        outputs=[
            emotion_output,
            confidence_output,
            drift_output,
            graph_output,
            mental_output
        ]
    )


# ----------------------------
# LAUNCH
# ----------------------------

demo.launch()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c494eb2d269878986.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [45]:
import inspect
from IPython.display import FileLink

# Get the source code of the Gradio app definition from the current notebook
# This assumes the Gradio app code is primarily within a single cell, here HfmnE2fgO2Zn
# For simplicity, we are capturing the content of the HfmnE2fgO2Zn cell.
# In a real scenario, you might need to combine multiple cells if the Gradio app is distributed.

gradio_app_code = """
import gradio as gr
import torch
import pickle
import plotly.graph_objects as go
import torch.nn.functional as F

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


# ----------------------------
# LOAD MODEL
# ----------------------------

tokenizer = DistilBertTokenizer.from_pretrained("emotion_model")
model = DistilBertForSequenceClassification.from_pretrained("emotion_model")

with open("label_encoder.pkl","rb") as f:
    le = pickle.load(f)


emotion_score = {
    "joy":3,
    "surprise":2,
    "neutral":1,
    "sadness":-1,
    "fear":-2,
    "anger":-3
}


# ----------------------------
# EMOTION PREDICTION
# ----------------------------

def predict_emotion(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs).item()

    confidence = torch.max(probs).item()

    emotion = le.inverse_transform([pred])[0]

    return emotion, confidence


# ----------------------------
# DRIFT DETECTION
# ----------------------------

def detect_drift(emotions):

    drift = []

    for i in range(1,len(emotions)):

        prev = emotion_score.get(emotions[i-1],0)
        curr = emotion_score.get(emotions[i],0)

        if abs(curr-prev) >= 2:
            drift.append(i+1)

    return drift


# ----------------------------
# MAIN ANALYSIS FUNCTION
# ----------------------------

def analyze_conversation(conversation):

    lines = conversation.split("\n")

    emotions = []
    confidence = []

    for line in lines:

        if line.strip() != "":
            emo, conf = predict_emotion(line)

            emotions.append(emo)
            confidence.append(round(conf*100,2))


    drift = detect_drift(emotions)

    scores = [emotion_score[e] for e in emotions]


    # Plot graph
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=list(range(1,len(scores)+1)),
        y=scores,
        mode="lines+markers",
        name="Emotion Trend"
    ))


    # Mental Health Analysis
    negative_count = sum(
        1 for e in emotions if e in ["sadness","fear","anger"]
    )

    if len(emotions) > 3 and negative_count >= len(emotions)/2:

        mental_status = '''
WARNING: Mental Health Alert

Your conversation shows a strong negative emotional trend.

Suggestions:
- Take a short break
- Talk to a trusted friend
- Try relaxation techniques
'''

    else:
        mental_status = "SUCCESS: Emotional state appears stable."


    return emotions, confidence, drift, fig, mental_status


# ----------------------------
# SAMPLE INPUTS
# ----------------------------

samples = {
"Happy → Angry Drift":
'''I was very happy today
My code worked perfectly
Suddenly my laptop crashed
Now I feel very angry''',

"Angry → Happy Recovery":
'''My project was not working
I felt very frustrated
My friend helped me debug
Now everything works
Now I feel very happy''',

"Stable Positive Mood":
'''Today was a great day
I enjoyed working on my project
Everything went smoothly
I feel satisfied''',

"Mental Stress Pattern":
'''I feel very sad today
Nothing seems to work
I feel anxious about my future
Everything feels overwhelming''',

"Mixed Emotion Conversation":
'''I was excited to start my project
But I faced many bugs
I got very angry and frustrated
Later I solved the issue
Now I feel relieved'''
}


# ----------------------------
# GRADIO UI
# ----------------------------

with gr.Blocks(title="Emotion Drift Detection") as demo:

    gr.Markdown("# Emotion Drift Detection Chatbot")

    gr.Markdown("Analyze emotional patterns and detect emotional drift in conversations.")

    sample_choice = gr.Dropdown(
        list(samples.keys()),
        label="Choose Sample Test"
    )

    sample_text = gr.Textbox(
        label="Sample Conversation",
        lines=6
    )

    sample_choice.change(
        lambda x: samples[x],
        sample_choice,
        sample_text
    )


    conversation_input = gr.Textbox(
        label="Enter Conversation (one sentence per line)",
        lines=8
    )

    analyze_btn = gr.Button("Analyze Emotion")


    emotion_output = gr.JSON(label="Predicted Emotions")

    confidence_output = gr.JSON(label="Confidence (%)")

    drift_output = gr.JSON(label="Emotion Drift Points")

    graph_output = gr.Plot(label="Emotion Trend Graph")

    mental_output = gr.Textbox(label="Mental Health Analysis")


    analyze_btn.click(
        analyze_conversation,
        inputs=conversation_input,
        outputs=[
            emotion_output,
            confidence_output,
            drift_output,
            graph_output,
            mental_output
        ]
    )


# ----------------------------
# LAUNCH
# ----------------------------

demo.launch()
"""

# Write the Gradio app code to a file
file_name = "appy1.py"
with open(file_name, "w") as f:
    f.write(gradio_app_code)

# Provide a download link
display(FileLink(file_name))

/content/appy1.py